# 03 – Privacy Demo (Governance Officer)

This notebook is the **Governance Officer** contribution for the project rubric.

It demonstrates, using the *real dataset loaded at runtime*:

1. **PII inventory** (what PII exists and why it matters)
2. **Pseudonymization** (deterministic salted hashing) for at least one PII field
3. **Data minimization** (reduce identifiability + minimize sensitive behavioral detail)
4. **Audit trail** (traceability of governance actions)
5. **GDPR mapping** to specific articles + **EU AI Act** note for credit scoring (academic mapping, not legal advice)

All outputs are written to the `output/` folder at the project root.


## 1) Imports

In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd


## 2) Robust dataset loading (root-level notebook)

The notebook is stored at the **project root**. This loader searches for dataset files in the required priority order:

1. `cleaned_credit_data.csv`
2. `cleaned_credit_applications.json`
3. `raw_credit_applications.json`

It prints the loaded file and dataset shape.

In [ ]:
PROJECT_ROOT = Path.cwd()

DATASET_PRIORITY = [
    "cleaned_credit_data.csv",
    "cleaned_credit_applications.json",
    "raw_credit_applications.json",
]

# We only use notebooks/ as a fallback (some teams keep files there)
SEARCH_DIRS = [
    PROJECT_ROOT,
    PROJECT_ROOT / "data",
    PROJECT_ROOT / "datasets",
    PROJECT_ROOT / "notebooks",  # fallback only
]

def find_dataset_file() -> Path:
    for filename in DATASET_PRIORITY:
        for d in SEARCH_DIRS:
            candidate = d / filename
            if candidate.exists():
                return candidate
    raise FileNotFoundError(
        "Could not find any dataset file. Looked for (in priority order): "
        f"{DATASET_PRIORITY} in dirs: {SEARCH_DIRS}"
    )

dataset_path = find_dataset_file()
print("Loaded source file:", dataset_path.as_posix())

def load_any_dataset(path: Path) -> pd.DataFrame:
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    # JSON: support list of records OR dict with nested records
    with path.open("r", encoding="utf-8") as f:
        obj = json.load(f)

    records = None
    if isinstance(obj, list):
        records = obj
    elif isinstance(obj, dict):
        # try common patterns: {"records": [...]}, {"data":[...]}, etc.
        for k in ["records", "data", "items", "applications"]:
            if k in obj and isinstance(obj[k], list):
                records = obj[k]
                break
        # otherwise choose first list value in dict
        if records is None:
            for v in obj.values():
                if isinstance(v, list):
                    records = v
                    break
        # if still None, treat whole dict as a single record
        if records is None:
            records = [obj]
    else:
        records = [obj]

    # Flatten nested JSON consistently
    df_json = pd.json_normalize(records, sep=".")
    return df_json

df = load_any_dataset(dataset_path)
print(f"Rows: {df.shape[0]:,}  Columns: {df.shape[1]:,}")
df.head(3)


## 3) PII inventory

This section creates a **PII inventory** table with:

- `classification` (direct/quasi/sensitive/high-risk)
- `gdpr_category` (identifier / online / demographic / profiling / automated decision)
- `risk_level` (High/Medium/Low)
- `present_in_data` and completeness metrics
- `governance_note` (what we should do about it)

We explicitly include key fields even if they are missing from a particular file.

In [ ]:
# Key fields we must classify even if missing
KEY_FIELDS = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code",
    "applicant_info.gender",
    "spending_behavior",
    "decision.loan_approved",
    "decision.rejection_reason",
    "decision.interest_rate",
    "decision.approved_amount",
    "financials.annual_income",
]

FIELD_POLICY = {
    "applicant_info.full_name": {
        "classification": "Direct identifier",
        "gdpr_category": "Personal data (identifier)",
        "risk_level": "High",
        "governance_note": "Pseudonymize; do not use in modeling; restrict access.",
    },
    "applicant_info.email": {
        "classification": "Direct identifier (contact)",
        "gdpr_category": "Personal data (contact)",
        "risk_level": "High",
        "governance_note": "Pseudonymize; store separately; restrict access.",
    },
    "applicant_info.ssn": {
        "classification": "National identifier",
        "gdpr_category": "Personal data (unique identifier)",
        "risk_level": "High",
        "governance_note": "Strongly protect; pseudonymize; never expose; consider tokenization.",
    },
    "applicant_info.ip_address": {
        "classification": "Online identifier",
        "gdpr_category": "Personal data (online identifier)",
        "risk_level": "High",
        "governance_note": "Pseudonymize; consider truncation; protect in logs.",
    },
    "applicant_info.date_of_birth": {
        "classification": "Quasi-identifier",
        "gdpr_category": "Personal data (demographic)",
        "risk_level": "Medium",
        "governance_note": "Minimize to age band; avoid storing full DOB in operational dataset.",
    },
    "applicant_info.zip_code": {
        "classification": "Quasi-identifier / proxy",
        "gdpr_category": "Personal data (location)",
        "risk_level": "Medium",
        "governance_note": "Minimize to ZIP3 (or similar) to reduce re-identification risk.",
    },
    "applicant_info.gender": {
        "classification": "Protected attribute",
        "gdpr_category": "Personal data (demographic)",
        "risk_level": "Medium",
        "governance_note": "Do not keep in operational dataset; retain only in controlled fairness/audit environment if needed.",
    },
    "spending_behavior": {
        "classification": "Behavioral profiling input",
        "gdpr_category": "Profiling / behavioral data",
        "risk_level": "High",
        "governance_note": "Minimize to aggregates; treat sensitive categories carefully; document purpose/necessity.",
    },
    "decision.loan_approved": {
        "classification": "Automated decision output",
        "gdpr_category": "Decision data (profiling outcome)",
        "risk_level": "High",
        "governance_note": "Maintain audit trail; enable human review; store model version and rationale.",
    },
    "decision.rejection_reason": {
        "classification": "Decision rationale",
        "gdpr_category": "Decision explanation data",
        "risk_level": "High",
        "governance_note": "Maintain traceability; ensure explanations are accurate and reviewable.",
    },
    "decision.interest_rate": {
        "classification": "Decision output",
        "gdpr_category": "Decision data (profiling outcome)",
        "risk_level": "Medium",
        "governance_note": "Maintain traceability and human oversight when algorithmic.",
    },
    "decision.approved_amount": {
        "classification": "Decision output",
        "gdpr_category": "Decision data (profiling outcome)",
        "risk_level": "Medium",
        "governance_note": "Maintain traceability and human oversight when algorithmic.",
    },
    "financials.annual_income": {
        "classification": "Financial attribute",
        "gdpr_category": "Personal data (financial)",
        "risk_level": "Medium",
        "governance_note": "Limit access; use only as necessary; document purpose and retention.",
    },
}

all_columns = sorted(set(df.columns).union(KEY_FIELDS))

rows = []
for col in all_columns:
    present = col in df.columns
    non_null = int(df[col].notna().sum()) if present else 0
    total = int(df.shape[0]) if present else int(df.shape[0])
    non_null_pct = (non_null / total * 100) if total > 0 else 0.0

    policy = FIELD_POLICY.get(col, None)
    if policy:
        classification = policy["classification"]
        gdpr_category = policy["gdpr_category"]
        risk_level = policy["risk_level"]
        note = policy["governance_note"]
    else:
        classification = "Other / business data"
        gdpr_category = "Not classified"
        risk_level = "Low"
        note = "Review if used for profiling or if it can identify a person when combined."

    rows.append({
        "column_name": col,
        "classification": classification,
        "gdpr_category": gdpr_category,
        "risk_level": risk_level,
        "present_in_data": bool(present),
        "non_null_count": non_null,
        "non_null_pct": round(non_null_pct, 2),
        "governance_note": note,
    })

pii_inventory = pd.DataFrame(rows)

# Show key fields first, then the rest
key_order = {name: i for i, name in enumerate(KEY_FIELDS)}
pii_inventory["__sort"] = pii_inventory["column_name"].map(lambda x: key_order.get(x, 10_000))
pii_inventory = pii_inventory.sort_values(["__sort", "risk_level", "column_name"]).drop(columns="__sort")

pii_inventory.head(25)


## 4) Pseudonymization (deterministic salted hashing)

We pseudonymize (when present):

- **Required:** `applicant_info.ssn`, `applicant_info.email`
- **If present:** `applicant_info.full_name`, `applicant_info.ip_address`

Rules:
- Salt comes from env var **`PSEUDONYM_SALT`**
- If missing, we use a visible demo fallback and print a warning
- We **do not overwrite** original columns; we add new `*_pseudo` columns


In [ ]:
PSEUDONYM_SALT = os.getenv("PSEUDONYM_SALT")
if not PSEUDONYM_SALT:
    PSEUDONYM_SALT = "DEMO_SALT_CHANGE_ME"
    print("WARNING: using demo salt; set PSEUDONYM_SALT in production")

def salted_hash(value: object) -> str | None:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    s = str(value).strip()
    if s == "":
        return None
    raw = f"{PSEUDONYM_SALT}::{s}".encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:16]

PSEUDO_TARGETS = {
    "applicant_info.ssn": "ssn",
    "applicant_info.email": "email",
    "applicant_info.full_name": "name",
    "applicant_info.ip_address": "ip",
}

df_pseudo = df.copy()
pseudonymized_columns = []

for col, prefix in PSEUDO_TARGETS.items():
    if col in df_pseudo.columns:
        new_col = f"{col}_pseudo"
        if col == "applicant_info.email":
            df_pseudo[new_col] = df_pseudo[col].apply(lambda x: f"{prefix}_{salted_hash(str(x).lower())}")
        else:
            df_pseudo[new_col] = df_pseudo[col].apply(lambda x: f"{prefix}_{salted_hash(x)}")
        pseudonymized_columns.append(new_col)

# Show a small preview (few rows only) of original vs pseudonymized
preview_cols = []
for col in PSEUDO_TARGETS.keys():
    if col in df_pseudo.columns and f"{col}_pseudo" in df_pseudo.columns:
        preview_cols.extend([col, f"{col}_pseudo"])

df_pseudo[preview_cols].head(5)


## 5) Data minimization

We create a minimized dataset for typical **operational** analytics by:

- Converting **DOB → `age_band`** (generalization)
- Converting **ZIP → `zip3`** (generalization)
- Converting **spending behavior → aggregate features** (minimize sensitive behavioral details)
- Dropping direct identifiers and raw detailed fields

**Gender handling:** we **drop** it from the operational minimized dataset. If gender is required for fairness auditing, it should be retained only in a **separately controlled audit environment** (restricted access, purpose-limited).

In [ ]:
SENSITIVE_SPENDING_CATEGORIES = {"Alcohol", "Gambling", "Adult Entertainment"}

def zip_to_zip3(val: object) -> str | None:
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return None
    raw = str(val).strip().replace(".0", "")
    digits = re.sub(r"\D", "", raw)
    return digits[:3] if len(digits) >= 3 else None

def age_band_from_dob(dob: object) -> str | None:
    if dob is None or (isinstance(dob, float) and pd.isna(dob)):
        return None
    dt = pd.to_datetime(dob, errors="coerce")
    if pd.isna(dt):
        return None
    today = pd.Timestamp.utcnow().tz_localize(None)
    age = (today - dt).days / 365.25
    if age < 25:
        return "18-24"
    if age < 35:
        return "25-34"
    if age < 45:
        return "35-44"
    if age < 55:
        return "45-54"
    if age < 65:
        return "55-64"
    return "65+"

def parse_spending(cell: object) -> list[dict]:
    if cell is None or (isinstance(cell, float) and pd.isna(cell)):
        return []
    if isinstance(cell, list):
        return cell
    if isinstance(cell, str):
        try:
            parsed = ast.literal_eval(cell)
            return parsed if isinstance(parsed, list) else []
        except Exception:
            return []
    return []

def spending_features(cell: object) -> dict:
    items = parse_spending(cell)
    total = 0.0
    cats = []
    sensitive = False
    for it in items:
        cat = it.get("category")
        amt = it.get("amount", 0)
        if cat:
            cats.append(cat)
            if cat in SENSITIVE_SPENDING_CATEGORIES:
                sensitive = True
        try:
            total += float(amt)
        except Exception:
            pass
    return {
        "spending_total_amount": round(total, 2),
        "spending_category_count": len(cats),
        "has_sensitive_spending": bool(sensitive),
    }

df_min = df_pseudo.copy()

if "applicant_info.date_of_birth" in df_min.columns:
    df_min["age_band"] = df_min["applicant_info.date_of_birth"].apply(age_band_from_dob)

if "applicant_info.zip_code" in df_min.columns:
    df_min["zip3"] = df_min["applicant_info.zip_code"].apply(zip_to_zip3)

if "spending_behavior" in df_min.columns:
    feats = df_min["spending_behavior"].apply(spending_features).apply(pd.Series)
    df_min = pd.concat([df_min, feats], axis=1)

# Drop direct/raw fields from minimized dataset
DROP_COLS = [
    "applicant_info.full_name",
    "applicant_info.email",
    "applicant_info.ssn",
    "applicant_info.ip_address",
    "applicant_info.date_of_birth",
    "applicant_info.zip_code",
    "spending_behavior",
    "notes",
    "applicant_info.gender",  # dropped from operational dataset; keep only in controlled fairness/audit environment if needed
]

minimized_dropped = [c for c in DROP_COLS if c in df_min.columns]
df_min = df_min.drop(columns=minimized_dropped)

df_min.head(3)


## 6) Governance controls summary (for rubric)

We record which governance controls are implemented in code vs proposed as policy controls.
This is written to:

- `output/governance_controls_summary.json`
- `output/privacy_governance_summary.csv`


In [ ]:
governance_controls = [
    {
        "control_name": "PII inventory",
        "purpose": "Identify direct identifiers, quasi-identifiers, profiling inputs, and decision outputs.",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 25", "Art. 5(1)(c)"],
        "related_ai_act_obligation": "Data governance & documentation",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Pseudonymization",
        "purpose": "Reduce re-identification risk by hashing identifiers with a salt.",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 25", "Art. 32"],
        "related_ai_act_obligation": "Record-keeping / traceability support",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Data minimization",
        "purpose": "Reduce granularity (DOB→age band, ZIP→ZIP3) and minimize sensitive behavioral detail.",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 5(1)(c)", "Art. 25"],
        "related_ai_act_obligation": "Data governance",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Audit trail",
        "purpose": "Traceability of processing actions (dataset loaded, transformations applied, outputs written).",
        "implemented_in_code": True,
        "status": "implemented",
        "related_gdpr_articles": ["Art. 25", "Art. 32"],
        "related_ai_act_obligation": "Logging / record-keeping",
        "owner": "Governance Officer",
    },
    {
        "control_name": "Consent tracking",
        "purpose": "Capture consent state where applicable; document lawful basis where consent is not used.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 6", "Art. 25"],
        "related_ai_act_obligation": "Governance documentation",
        "owner": "Data Controller / Governance",
    },
    {
        "control_name": "Retention policy",
        "purpose": "Define and enforce retention schedule for raw PII and derived datasets.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 5(1)(e)", "Art. 25"],
        "related_ai_act_obligation": "Record-keeping & lifecycle management",
        "owner": "Governance + Data Engineering",
    },
    {
        "control_name": "Human oversight",
        "purpose": "Human review for automated credit decisions and overrides with documentation.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 22"],
        "related_ai_act_obligation": "Human oversight",
        "owner": "Operations / Risk Team",
    },
    {
        "control_name": "Right-to-erasure workflow",
        "purpose": "Operational process to locate and delete personal data upon request.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 17"],
        "related_ai_act_obligation": "Governance / record-keeping",
        "owner": "Data Controller / Governance",
    },
    {
        "control_name": "Model/version logging",
        "purpose": "Record model version, decision source, and feature snapshot reference for traceability.",
        "implemented_in_code": False,
        "status": "proposed",
        "related_gdpr_articles": ["Art. 22", "Art. 25"],
        "related_ai_act_obligation": "Logging / traceability",
        "owner": "ML Engineering / Governance",
    },
]

controls_df = pd.DataFrame(governance_controls)
controls_df


## 7) Audit trail 

We log key events with:
- `run_id`, `timestamp_utc`, `action`, `details`
- `dataset_source`
- `record_count`

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("privacy_%Y%m%dT%H%M%SZ")

audit_trail = []

def audit(action: str, details: dict) -> None:
    audit_trail.append({
        "run_id": RUN_ID,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "action": action,
        "details": details,
        "dataset_source": dataset_path.name,
        "record_count": int(df.shape[0]),
    })

# Events
audit("dataset_loaded", {
    "path": dataset_path.as_posix(),
    "rows": int(df.shape[0]),
    "columns": int(df.shape[1]),
})

audit("pii_inventory_created", {
    "key_fields_present": [c for c in KEY_FIELDS if c in df.columns],
    "high_risk_fields_present": [c for c in ["spending_behavior", "decision.loan_approved"] if c in df.columns],
})

audit("pseudonymization_applied", {
    "pseudonymized_columns": pseudonymized_columns,
})

audit("data_minimization_applied", {
    "dropped_columns": minimized_dropped,
    "derived_columns": [c for c in ["age_band", "zip3", "spending_total_amount", "spending_category_count", "has_sensitive_spending"] if c in df_min.columns],
})

audit_trail[:3]


## 8) Write outputs

This notebook preserves existing outputs and adds two new governance outputs.

**Existing outputs:**
- `output/pii_inventory.csv`
- `output/credit_data_pseudonymized.csv`
- `output/credit_data_minimized.csv`
- `output/audit_trail.json`

**New outputs:**
- `output/governance_controls_summary.json`
- `output/privacy_governance_summary.csv`


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# Preserve existing outputs
pii_inventory.to_csv(OUTPUT_DIR / "pii_inventory.csv", index=False)
df_pseudo.to_csv(OUTPUT_DIR / "credit_data_pseudonymized.csv", index=False)
df_min.to_csv(OUTPUT_DIR / "credit_data_minimized.csv", index=False)

with (OUTPUT_DIR / "audit_trail.json").open("w", encoding="utf-8") as f:
    json.dump(audit_trail, f, indent=2, ensure_ascii=False)

# New governance outputs
with (OUTPUT_DIR / "governance_controls_summary.json").open("w", encoding="utf-8") as f:
    json.dump(governance_controls, f, indent=2, ensure_ascii=False)

# A small CSV summary that is easy to show in a viva/Q&A
summary_rows = [
    {"metric": "source_file_used", "value": dataset_path.name},
    {"metric": "rows_processed", "value": int(df.shape[0])},
    {"metric": "columns_processed", "value": int(df.shape[1])},
    {"metric": "key_pii_fields_present", "value": ", ".join([c for c in KEY_FIELDS if c in df.columns])},
    {"metric": "pseudonymized_columns", "value": ", ".join(pseudonymized_columns)},
    {"metric": "minimized_columns_dropped", "value": ", ".join(minimized_dropped)},
    {"metric": "outputs_written", "value": ", ".join([
        "pii_inventory.csv",
        "credit_data_pseudonymized.csv",
        "credit_data_minimized.csv",
        "audit_trail.json",
        "governance_controls_summary.json",
        "privacy_governance_summary.csv",
    ])},
]
privacy_governance_summary = pd.DataFrame(summary_rows)
privacy_governance_summary.to_csv(OUTPUT_DIR / "privacy_governance_summary.csv", index=False)

audit("governance_outputs_written", {"output_dir": OUTPUT_DIR.as_posix()})

print("Wrote outputs to:", OUTPUT_DIR.as_posix())
for fname in [
    "pii_inventory.csv",
    "credit_data_pseudonymized.csv",
    "credit_data_minimized.csv",
    "audit_trail.json",
    "governance_controls_summary.json",
    "privacy_governance_summary.csv",
]:
    print("-", (OUTPUT_DIR / fname).as_posix())


## 9) GDPR mapping (academic) + EU AI Act note

This section maps what we observed and implemented to **specific GDPR articles** (academic mapping, not legal advice):

- **Art. 6 (Lawful basis):** processing must have a lawful basis (e.g., contract, legitimate interest). This must be documented by the controller; code can capture metadata fields (lawful basis label).
- **Art. 5(1)(c) (Data minimization):** we drop direct identifiers (name/email/SSN/IP) from the operational dataset and reduce granularity (DOB→age band, ZIP→ZIP3).
- **Art. 5(1)(e) (Storage limitation):** retention periods should be defined and enforced (e.g., delete raw PII after X days; keep minimized aggregates longer). This notebook recommends policy + metadata fields for retention.
- **Art. 17 (Right to erasure):** an erasure workflow should locate and delete personal data on request. Pseudonymized identifiers help locate records without exposing raw identifiers.
- **Art. 22 (Automated decision-making):** credit approval outcomes and rejection reasons indicate automated decision outputs. Governance should support **human review**, the ability to contest decisions, and documentation of overrides.
- **Art. 25 (Data protection by design and default):** apply privacy controls early (PII inventory, pseudonymization, minimization) and default to minimized operational datasets.
- **Art. 32 (Security of processing):** pseudonymization supports security, but production also requires encryption, access controls, monitoring, and secure key/salt management.

### EU AI Act (academic note)
Credit scoring / creditworthiness assessment is widely treated as a **high-risk AI use case**, so governance expectations include:
- **logging and record-keeping**
- **traceability**
- **human oversight**
- **data governance and quality management**
- **documentation of the system and decisions**

This notebook implements baseline logging/traceability (audit trail + outputs) and recommends governance metadata and oversight processes.


## 10) Final run summary

This final cell prints exactly what was done and which files were produced.

In [ ]:
detected_pii = pii_inventory[
    (pii_inventory["present_in_data"] == True) &
    (pii_inventory["risk_level"].isin(["High", "Medium"]))
]["column_name"].tolist()

print("=== Privacy Demo Summary ===")
print("Source file used:", dataset_path.name)
print("Rows processed:", int(df.shape[0]))
print("Columns processed:", int(df.shape[1]))
print()

print("PII / high-risk fields detected (present + Medium/High risk):")
for c in detected_pii:
    print("-", c)

print()
print("Pseudonymized columns created:")
for c in pseudonymized_columns:
    print("-", c)

print()
print("Minimization dropped columns:")
for c in minimized_dropped:
    print("-", c)

print()
print("Outputs written:")
for fname in [
    "pii_inventory.csv",
    "credit_data_pseudonymized.csv",
    "credit_data_minimized.csv",
    "audit_trail.json",
    "governance_controls_summary.json",
    "privacy_governance_summary.csv",
]:
    print("-", (OUTPUT_DIR / fname).as_posix())
